# Tutorial: Clasificación de abandono de empleado (Employee Churn Model)

## Contexto del ejercicio
En este notebook construiremos un modelo de clasificación para estimar la probabilidad de que un empleado abandone la organización. Este tipo de ejercicio es útil en iniciativas de *People Analytics*, retención de talento y análisis predictivo aplicado a recursos humanos.

## Audiencia
- Alumnos de posgrado y educación continua que desean practicar clasificación supervisada con datos tabulares.

## Prerrequisitos
- Conocimientos básicos de Python y pandas.
- Nociones generales de entrenamiento y evaluación de modelos de clasificación.

## Objetivo
Construir un modelo base de **abandono de empleado** usando `scikit-learn`, evaluar su desempeño e interpretar qué variables se relacionan con la salida del personal.


## Ruta del ejercicio
1. Preparar dependencias y entorno.
2. Cargar el dataset en Google Colab o desde GitHub.
3. Explorar la estructura del problema y la variable objetivo.
4. Preparar variables numéricas y categóricas.
5. Entrenar un modelo de clasificación con `LogisticRegression`.
6. Evaluar el modelo con métricas y matriz de confusión.
7. Interpretar variables relevantes y discutir implicaciones de negocio.


## Cómo usar el dataset en Google Colab
Tiene dos opciones:
- **Opción 1:** usar automáticamente el archivo publicado en GitHub.
- **Opción 2:** subir manualmente el CSV a Colab desde su computadora.

En este notebook la opción predeterminada es usar GitHub, pero puede activar la carga manual cambiando `USE_COLAB_UPLOAD = True`.


In [ ]:
import sys

if 'google.colab' in sys.modules:
    try:
        import sklearn  # noqa: F401
    except ImportError:
        %pip install -q scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
RAW_DATA_URL = 'https://raw.githubusercontent.com/JuanBaldemarG/portafoliocolabJBGV/main/data/employee-churn/HR_dataset_copy.csv'
USE_COLAB_UPLOAD = False
LOCAL_CANDIDATES = [
    Path('../data/employee-churn/HR_dataset_copy.csv'),
    Path('data/employee-churn/HR_dataset_copy.csv'),
    Path('/content/HR_dataset_copy.csv')
]

def load_dataset() -> pd.DataFrame:
    if 'google.colab' in sys.modules:
        if USE_COLAB_UPLOAD:
            from google.colab import files
            uploaded = files.upload()
            uploaded_name = next(iter(uploaded))
            return pd.read_csv(uploaded_name)

        for candidate in LOCAL_CANDIDATES:
            if candidate.exists():
                return pd.read_csv(candidate)

        return pd.read_csv(RAW_DATA_URL)

    for candidate in LOCAL_CANDIDATES:
        if candidate.exists():
            return pd.read_csv(candidate)

    return pd.read_csv(RAW_DATA_URL)

df = load_dataset()
df.head()


## Exploración inicial
Antes de modelar, conviene entender cuántos registros tenemos, qué columnas existen y qué tipos de datos están presentes. Esto ayuda a decidir cómo preparar el pipeline.


In [ ]:
print(f'Registros: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
display(df.dtypes.to_frame('tipo'))


In [ ]:
target = 'left'
target_distribution = df[target].value_counts().sort_index()
target_share = df[target].value_counts(normalize=True).sort_index()
display(pd.DataFrame({'conteo': target_distribution, 'proporción': target_share}))

ax = target_distribution.plot(kind='bar', color=['#4c78a8', '#f58518'], figsize=(6, 4))
ax.set_title('Distribución de la variable objetivo: left')
ax.set_xlabel('Clase')
ax.set_ylabel('Número de empleados')
plt.show()


### Interpretación inicial
La variable `left` representa si el empleado salió de la organización (`1`) o permaneció (`0`). Esta distribución es importante porque una base muy desbalanceada puede afectar la interpretación de métricas como *accuracy*.


In [ ]:
summary_cols = ['satisfaction_level', 'last_evaluation', 'number_project', 'average_montly_hours', 'time_spend_company']
df.groupby(target)[summary_cols].mean().round(2)


## Preparación del modelo
Usaremos la variable `left` como objetivo. Las columnas numéricas y categóricas se transforman dentro de un `Pipeline` para dejar el flujo reproducible y compatible con Colab.


In [ ]:
X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Tamaño entrenamiento: {X_train.shape[0]:,}')
print(f'Tamaño prueba: {X_test.shape[0]:,}')
print(f'Variables numéricas: {len(numeric_features)}')
print(f'Variables categóricas: {len(categorical_features)}')


In [ ]:
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred)
roc_auc = roc_auc_score(y_test, proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'ROC AUC: {roc_auc:.4f}')
print()
print(classification_report(y_test, pred))


### Cómo interpretar estas métricas
- **Accuracy:** proporción total de predicciones correctas.
- **Precision:** de los empleados marcados como abandono, cuántos realmente abandonaron.
- **Recall:** de los empleados que realmente abandonaron, cuántos fueron detectados por el modelo.
- **ROC AUC:** capacidad general del modelo para distinguir entre permanencia y abandono a distintos umbrales.

En problemas de retención suele ser especialmente importante vigilar el **recall** de la clase de abandono, porque un falso negativo significa no detectar a tiempo a un empleado con riesgo de salida.


In [ ]:
cm = confusion_matrix(y_test, pred)
cm_df = pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1'])
display(cm_df)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión')
plt.show()


### Interpretación de la matriz de confusión
- **Pred 1 / Real 1:** empleados con abandono correctamente detectados.
- **Pred 0 / Real 1:** empleados que abandonaron pero el modelo no detectó.
- **Pred 1 / Real 0:** empleados marcados con riesgo aunque en realidad permanecieron.

Desde una perspectiva de negocio, los falsos negativos suelen ser más costosos si la organización quiere intervenir antes de perder talento clave.


In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]
coef_df = (
    pd.DataFrame({'feature': feature_names, 'coef': coefficients})
    .assign(abs_coef=lambda d: d['coef'].abs())
    .sort_values('abs_coef', ascending=False)
)
coef_df[['feature', 'coef']].head(12)


## Interpretación de variables relevantes
En una regresión logística, un coeficiente positivo empuja la predicción hacia `left = 1` y un coeficiente negativo hacia `left = 0`.

Esto no debe leerse como causalidad directa, sino como una señal estadística dentro de este dataset. Conviene complementar estos hallazgos con conocimiento del proceso, entrevistas y políticas de recursos humanos.


In [ ]:
top_positive = coef_df.sort_values('coef', ascending=False).head(5)[['feature', 'coef']]
top_negative = coef_df.sort_values('coef', ascending=True).head(5)[['feature', 'coef']]

print('Variables más asociadas con abandono (coeficientes positivos):')
display(top_positive)

print('Variables más asociadas con permanencia (coeficientes negativos):')
display(top_negative)


## Conclusión ejecutiva
Este notebook deja una línea base reproducible para el problema de abandono de empleado. A partir de los resultados obtenidos, el grupo puede discutir preguntas como:
- ¿Qué tan útil es el modelo para priorizar intervenciones de retención?
- ¿Conviene optimizar el modelo hacia mayor recall o mayor precisión?
- ¿Qué variables merecen revisión por parte del área de recursos humanos?

El siguiente paso natural sería comparar este modelo con alternativas como árboles, bosques aleatorios o *gradient boosting*.


## Ejercicio para el alumno
Pruebe una de estas extensiones:
1. Cambiar `LogisticRegression` por `RandomForestClassifier`.
2. Ajustar el umbral de clasificación usando `predict_proba`.
3. Comparar resultados quitando la columna categórica `functional area`.
4. Evaluar si la satisfacción del empleado parece ser una variable especialmente sensible.


In [ ]:
# Respuesta sugerida: use este espacio para probar un segundo modelo o un nuevo umbral.
# Ejemplo:
# from sklearn.ensemble import RandomForestClassifier
# ...


## Errores comunes y extensiones
**Error común:** olvidar el tratamiento de variables categóricas y pasar texto crudo al modelo.

**Error común:** quedarse solo con accuracy y no revisar recall, precisión o matriz de confusión.

**Extensión sugerida:** agregar validación cruzada, comparar varios clasificadores y documentar cuál conviene presentar como modelo final según el objetivo del negocio.
